# Lab 4 - Dataset Inspection & Feature Engineering EDA

This notebook explores the sampled Gold dataset before running it through the feature engineering pipeline. The goal is to understand the data distribution and figure out which features make sense to build.

We look at:
- Basic dataset stats (shape, columns, missing values)
- Rating distribution
- Review length distribution
- Why these matter for the features we chose

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

## 1. Load the Dataset

We load the sampled gold dataset that was registered as a data asset in Azure ML. If running locally, point this to a local copy of the parquet file.

In [ ]:
# Update this path to wherever your sampled gold parquet file is
# If running in Databricks, use the ADLS path instead
DATA_PATH = "data/features_v1_sampled/"

# Try loading - handle both single file and folder
if os.path.isdir(DATA_PATH):
    files = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet')]
    df = pd.concat([pd.read_parquet(os.path.join(DATA_PATH, f)) for f in files], ignore_index=True)
else:
    df = pd.read_parquet(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 2. Basic Inspection

Quick look at the data types, missing values, and first few rows. This helps us understand what we're working with before we start building features.

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print("Missing values:")
    print(missing)
else:
    print("No missing values found")

In [ ]:
df.describe()

## 3. Rating Distribution

Looking at how ratings are spread across the dataset. This matters for a couple reasons:
- If ratings are heavily skewed (e.g. mostly 5 stars), our sentiment features need to capture more subtle differences
- Class imbalance affects how we'd train any downstream model
- It tells us if the dataset is representative of real user behavior or if there's some sampling bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of rating counts
rating_counts = df['overall'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color=['#d32f2f', '#f57c00', '#fbc02d', '#7cb342', '#388e3c'])
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution')
axes[0].set_xticks([1, 2, 3, 4, 5])

# Percentage breakdown
rating_pct = (rating_counts / len(df) * 100).round(1)
axes[1].pie(rating_pct.values, labels=[f'{int(k)} stars\n({v}%)' for k, v in rating_pct.items()],
            colors=['#d32f2f', '#f57c00', '#fbc02d', '#7cb342', '#388e3c'],
            startangle=90)
axes[1].set_title('Rating Percentage Breakdown')

plt.tight_layout()
plt.show()

print("\nRating counts:")
print(rating_counts)
print(f"\nMean rating: {df['overall'].mean():.2f}")
print(f"Median rating: {df['overall'].median():.1f}")

**Why this matters for feature engineering:**

Amazon reviews tend to be skewed towards higher ratings - most people leave a review when they like something. This means:
- Simple features like average rating per product might not be very informative since most products cluster around 4-5 stars
- We need features that capture nuance in the text (sentiment, specific word patterns) to differentiate between a lukewarm 4-star and an enthusiastic 5-star review
- The compound sentiment score from VADER helps here because it gives a continuous value instead of just positive/negative

## 4. Review Length Distribution

How long are the reviews? This directly relates to the length features we compute in the pipeline.

In [ ]:
# Compute review lengths
df['_word_count'] = df['reviewText'].fillna('').apply(lambda x: len(x.split()))
df['_char_count'] = df['reviewText'].fillna('').apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count distribution
axes[0].hist(df['_word_count'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['_word_count'].median(), color='red', linestyle='--', label=f'Median: {df["_word_count"].median():.0f} words')
axes[0].axvline(df['_word_count'].mean(), color='orange', linestyle='--', label=f'Mean: {df["_word_count"].mean():.0f} words')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Review Length (Words)')
axes[0].legend()
axes[0].set_xlim(0, df['_word_count'].quantile(0.95))  # cut off extreme outliers for readability

# Character count distribution
axes[1].hist(df['_char_count'], bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(df['_char_count'].median(), color='red', linestyle='--', label=f'Median: {df["_char_count"].median():.0f} chars')
axes[1].axvline(df['_char_count'].mean(), color='orange', linestyle='--', label=f'Mean: {df["_char_count"].mean():.0f} chars')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Review Length (Characters)')
axes[1].legend()
axes[1].set_xlim(0, df['_char_count'].quantile(0.95))

plt.tight_layout()
plt.show()

print(f"Word count stats:")
print(df['_word_count'].describe().round(1))
print(f"\nReviews with < 10 chars: {(df['_char_count'] < 10).sum()} ({(df['_char_count'] < 10).mean()*100:.1f}%)")

**Why this matters for feature engineering:**

Review length is one of the simplest features but it carries real signal:
- Very short reviews are usually not helpful ("great product" or "bad") - that's why we filter under 10 chars in the normalize step
- Longer reviews tend to be more detailed and often come from people who feel strongly about the product
- There's usually a correlation between review length and helpfulness votes
- The right tail (very long reviews) might be outliers or copy-pasted content

We capture both word count and character count because they measure slightly different things - a review with lots of short words vs fewer long technical words.

## 5. Review Length vs Rating

Does the length of a review change depending on the rating? This helps us understand if length interacts with sentiment.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Box plot of word count by rating
data_by_rating = [df[df['overall'] == r]['_word_count'].values for r in sorted(df['overall'].unique())]
bp = ax.boxplot(data_by_rating, labels=[str(int(r)) for r in sorted(df['overall'].unique())],
                patch_artist=True, showfliers=False)

colors = ['#d32f2f', '#f57c00', '#fbc02d', '#7cb342', '#388e3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Rating')
ax.set_ylabel('Word Count')
ax.set_title('Review Length by Rating')

plt.tight_layout()
plt.show()

# Mean word count per rating
print("Mean word count by rating:")
print(df.groupby('overall')['_word_count'].mean().round(1))

**Why this matters:**

If we see that negative reviews tend to be longer (people complaining in detail) or shorter (just venting), thats useful information. It confirms that review length features will interact with sentiment features in interesting ways for downstream models. The combination of length + sentiment gives more signal than either alone.

## 6. Summary

Based on this exploration, here's what drove the feature engineering choices:

| Observation | Feature Decision |
|---|---|
| Ratings are skewed high | Need sentiment features to capture nuanced differences |
| Review lengths vary a lot | Include word/char count as basic but useful predictors |
| Short reviews lack info | Filter reviews < 10 chars in normalization |
| Length correlates with rating | Length + sentiment combo will be informative |
| Text contains noise (URLs, numbers) | Normalize text before extracting features |
| Words alone miss context | Add SBERT embeddings for semantic understanding |
| Some phrases matter more than single words | Use bigrams in TF-IDF (ngram_range 1,2) |

In [ ]:
# Cleanup temp columns
df.drop(columns=['_word_count', '_char_count'], inplace=True, errors='ignore')
print("Done - dataset inspection complete")